# Kaggle inference: model_topic

Runs the topic BERT model on `bert_candidates.parquet` and writes predictions plus metrics to `/kaggle/working`. The notebook is path-tolerant: it searches Kaggle inputs first, then local project paths.

### Шаг 1. Настройка окружения и путей

Импортируются базовые модули, определяется режим Kaggle/local и задаются имена входного parquet, папки модели, текстовой колонки, batch size и рабочей директории для результатов.


In [ ]:
from pathlib import Path
import gc
import json
import os
import shutil
import time
import warnings

warnings.filterwarnings('ignore')

IS_KAGGLE = Path('/kaggle/input').exists()
KAGGLE_INPUT = Path('/kaggle/input')
WORKING = Path('/kaggle/working') if IS_KAGGLE else Path('data/kaggle_working/topic')
WORKING.mkdir(parents=True, exist_ok=True)

INPUT_FILE_NAME = 'bert_candidates.parquet'
MODEL_DIR_NAME = 'model_topic'
TEXT_COL = 'text'
ID_COL = 'post_uid'
DEFAULT_THRESHOLD = 0.5
BATCH_SIZE_GPU = 32
BATCH_SIZE_CPU = 4

print('IS_KAGGLE:', IS_KAGGLE)
print('WORKING:', WORKING.resolve())


### Шаг 2. Проверка зависимостей

Проверяется наличие `torch`, `transformers`, `safetensors`, `pyarrow` и `tqdm`. Если в Kaggle-окружении чего-то нет, пакет устанавливается перед запуском инференса.


In [ ]:
import importlib.util
import subprocess
import sys

missing = [pkg for pkg in ['torch', 'transformers', 'safetensors', 'pyarrow', 'tqdm'] if importlib.util.find_spec(pkg) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())


### Шаг 3. Поиск входных файлов и нормализация модели

Ноутбук ищет `bert_candidates.parquet` и папку `model_topic` среди `/kaggle/input` и локальных fallback-путей. Файлы модели копируются в рабочую папку с обычными именами вроде `config.json` и `model.safetensors`, даже если исходно они назывались `config (1).json`.


In [ ]:
def find_file(file_name, local_fallbacks=()):
    candidates = []
    if IS_KAGGLE:
        candidates.extend(KAGGLE_INPUT.glob(f'**/{file_name}'))
    candidates.extend(Path.cwd().glob(f'**/{file_name}'))
    candidates.extend(Path(p) for p in local_fallbacks)
    for p in candidates:
        if p.exists() and p.is_file():
            return p.resolve()
    raise FileNotFoundError(f'Could not find {file_name}. Add it as a Kaggle dataset or put it in the project tree.')


def find_dir(dir_name, local_fallbacks=()):
    candidates = []
    if IS_KAGGLE:
        candidates.extend(KAGGLE_INPUT.glob(f'**/{dir_name}'))
    candidates.extend(Path.cwd().glob(f'**/{dir_name}'))
    candidates.extend(Path(p) for p in local_fallbacks)
    for p in candidates:
        if p.exists() and p.is_dir():
            return p.resolve()
    raise FileNotFoundError(f'Could not find directory {dir_name}. Add it as a Kaggle dataset or put it in the project tree.')


def first_match(src_dir, patterns):
    for pattern in patterns:
        hits = sorted(src_dir.glob(pattern))
        if hits:
            return hits[0]
    return None


def normalize_transformers_dir(src_dir, dst_dir, extra_config_patterns=()):
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    required = {
        'config.json': ['config.json', 'config*.json'],
        'tokenizer.json': ['tokenizer.json', 'tokenizer*.json'],
        'tokenizer_config.json': ['tokenizer_config.json', 'tokenizer_config*.json'],
    }
    for target, patterns in required.items():
        src = first_match(src_dir, patterns)
        if src is not None:
            shutil.copy2(src, dst_dir / target)

    weight = first_match(src_dir, ['model.safetensors', 'model*.safetensors', 'pytorch_model.bin', 'pytorch_model*.bin'])
    if weight is None:
        raise FileNotFoundError(f'No model weights found in {src_dir}. Expected model.safetensors or pytorch_model.bin.')
    shutil.copy2(weight, dst_dir / ('model.safetensors' if weight.suffix == '.safetensors' else 'pytorch_model.bin'))

    for name in ['special_tokens_map.json', 'vocab.txt', 'added_tokens.json']:
        src = src_dir / name
        if src.exists():
            shutil.copy2(src, dst_dir / name)

    copied_extra = []
    for pattern in extra_config_patterns:
        src = first_match(src_dir, [pattern])
        if src is not None:
            target = pattern.replace('*', '').replace(' (1)', '')
            if not target.endswith('.json'):
                target = src.name.replace(' (1)', '')
            shutil.copy2(src, dst_dir / target)
            copied_extra.append(dst_dir / target)

    return dst_dir, copied_extra

input_path = find_file(INPUT_FILE_NAME, ['data/prefilter/bert_candidates.parquet'])
model_src = find_dir(MODEL_DIR_NAME, ['model_topic'])
model_dir, extra = normalize_transformers_dir(model_src, WORKING / 'normalized_model_topic', ['topic_model_config*.json'])

print('INPUT:', input_path)
print('MODEL_SRC:', model_src)
print('MODEL_DIR:', model_dir)
print('EXTRA:', extra)


### Шаг 4. Загрузка parquet-кандидатов

Загружается итоговый файл предфильтра `bert_candidates.parquet`, проверяется наличие текстовой колонки и идентификатора поста. На выходе получается датафрейм постов, которые нужно разметить BERT-моделью по темам.


In [ ]:
df = pd.read_parquet(input_path)
if TEXT_COL not in df.columns:
    raise ValueError(f'Missing text column: {TEXT_COL}')
if ID_COL not in df.columns:
    df[ID_COL] = np.arange(len(df)).astype(str)

df[TEXT_COL] = df[TEXT_COL].fillna('').astype(str)
print('rows:', len(df))
print('columns:', list(df.columns))
df[[ID_COL, TEXT_COL]].head(3)


### Шаг 5. Чтение конфигурации тем и порогов

Из `topic_model_config.json` и `config.json` извлекаются список тематических меток, максимальная длина токенизации и индивидуальные thresholds для каждой темы.


In [ ]:
topic_cfg_path = model_dir / 'topic_model_config.json'
topic_cfg = json.loads(topic_cfg_path.read_text(encoding='utf-8')) if topic_cfg_path.exists() else {}
thresholds = topic_cfg.get('thresholds', {})
max_length = int(topic_cfg.get('max_length', 512))

with open(model_dir / 'config.json', 'r', encoding='utf-8') as f:
    hf_config = json.load(f)
labels = [hf_config['id2label'][str(i)] for i in range(len(hf_config['id2label']))]
threshold_arr = np.array([float(thresholds.get(label, DEFAULT_THRESHOLD)) for label in labels], dtype=np.float32)

print('labels:', labels)
print('thresholds:', dict(zip(labels, threshold_arr.tolist())))
print('max_length:', max_length)


### Шаг 6. Загрузка tokenizer и model_topic

Модель и tokenizer загружаются локально из нормализованной папки. Если доступна GPU, модель переводится на CUDA и использует `float16` для экономии памяти.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = BATCH_SIZE_GPU if device.type == 'cuda' else BATCH_SIZE_CPU

tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
model_kwargs = {'local_files_only': True}
if device.type == 'cuda':
    model_kwargs['torch_dtype'] = torch.float16
model = AutoModelForSequenceClassification.from_pretrained(model_dir, **model_kwargs)
model.to(device)
model.eval()

print('device:', device)
print('batch_size:', batch_size)


### Шаг 7. Инференс по темам

Тексты батчами токенизируются и прогоняются через модель. Логиты преобразуются через sigmoid в вероятности по каждой теме.


In [ ]:
texts = df[TEXT_COL].tolist()
all_probs = []
started = time.time()

for start in tqdm(range(0, len(texts), batch_size), desc='topic inference'):
    batch_texts = texts[start:start + batch_size]
    encoded = tokenizer(
        batch_texts,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors='pt',
    )
    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.inference_mode():
        logits = model(**encoded).logits
        probs = torch.sigmoid(logits).detach().float().cpu().numpy()
    all_probs.append(probs)

probs = np.vstack(all_probs) if all_probs else np.zeros((0, len(labels)), dtype=np.float32)
preds = probs >= threshold_arr[None, :]
print('done in seconds:', round(time.time() - started, 2))
print('probs shape:', probs.shape)


### Шаг 8. Формирование файла с предсказаниями

К исходным постам добавляются вероятности и бинарные предсказания по `t1..t5`, а также агрегаты `bert_topic_any`, `bert_topic_count`, `bert_topic_labels`. Результат сохраняется в parquet и csv.


In [ ]:
result = df.copy()
for j, label in enumerate(labels):
    topic = label.replace('_relevant', '')
    result[f'prob_{label}'] = probs[:, j].astype('float32')
    result[f'pred_{label}'] = preds[:, j].astype('int8')
    result[f'prob_{topic}'] = probs[:, j].astype('float32')
    result[f'pred_{topic}'] = preds[:, j].astype('int8')

short_topics = [label.replace('_relevant', '') for label in labels]
result['bert_topic_any'] = preds.any(axis=1).astype('int8')
result['bert_topic_count'] = preds.sum(axis=1).astype('int8')
result['bert_topic_labels'] = [';'.join([short_topics[j] for j, ok in enumerate(row) if ok]) for row in preds]

out_parquet = WORKING / 'bert_candidates_topic_predictions.parquet'
out_csv = WORKING / 'bert_candidates_topic_predictions.csv'
result.to_parquet(out_parquet, index=False, compression='zstd')
result.to_csv(out_csv, index=False)
print('saved:', out_parquet)
print('saved:', out_csv)
result[[ID_COL, 'bert_topic_any', 'bert_topic_count', 'bert_topic_labels'] + [f'prob_{t}' for t in short_topics]].head()


### Шаг 9. Подсчет метрик и распределений

Считаются агрегаты по темам: сколько постов получило каждую метку, средние и квантильные вероятности, общая доля размеченных. Если в файле есть ручные labels, дополнительно считаются precision, recall, F1 и PR-AUC.


In [ ]:
def safe_float(x):
    return None if pd.isna(x) else float(x)

label_rows = []
for j, label in enumerate(labels):
    topic = label.replace('_relevant', '')
    p = probs[:, j]
    y = preds[:, j]
    label_rows.append({
        'label': label,
        'topic': topic,
        'threshold': float(threshold_arr[j]),
        'pred_positive': int(y.sum()),
        'pred_positive_rate': float(y.mean()) if len(y) else 0.0,
        'prob_min': safe_float(np.min(p)) if len(p) else None,
        'prob_mean': safe_float(np.mean(p)) if len(p) else None,
        'prob_median': safe_float(np.median(p)) if len(p) else None,
        'prob_p90': safe_float(np.quantile(p, 0.90)) if len(p) else None,
        'prob_p95': safe_float(np.quantile(p, 0.95)) if len(p) else None,
        'prob_max': safe_float(np.max(p)) if len(p) else None,
    })
label_distribution = pd.DataFrame(label_rows)
label_distribution.to_csv(WORKING / 'topic_label_distribution.csv', index=False)

metrics = {
    'input_path': str(input_path),
    'model_path': str(model_src),
    'rows': int(len(result)),
    'labels': labels,
    'thresholds': {label: float(threshold_arr[i]) for i, label in enumerate(labels)},
    'bert_topic_any_count': int(result['bert_topic_any'].sum()),
    'bert_topic_any_rate': float(result['bert_topic_any'].mean()) if len(result) else 0.0,
    'mean_topic_count': float(result['bert_topic_count'].mean()) if len(result) else 0.0,
    'ground_truth_available': all(label in result.columns for label in labels),
}

if metrics['ground_truth_available']:
    from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score
    y_true = result[labels].fillna(0).astype(int).values
    y_pred = preds.astype(int)
    metrics.update({
        'micro_precision': float(precision_score(y_true, y_pred, average='micro', zero_division=0)),
        'micro_recall': float(recall_score(y_true, y_pred, average='micro', zero_division=0)),
        'micro_f1': float(f1_score(y_true, y_pred, average='micro', zero_division=0)),
        'macro_precision': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
        'macro_recall': float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
    })
    try:
        metrics['macro_pr_auc'] = float(average_precision_score(y_true, probs, average='macro'))
        metrics['micro_pr_auc'] = float(average_precision_score(y_true, probs, average='micro'))
    except Exception as exc:
        metrics['pr_auc_error'] = repr(exc)

metrics_path = WORKING / 'topic_metrics.json'
metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(metrics, ensure_ascii=False, indent=2))
label_distribution


### Шаг 10. Очистка памяти и список результатов

Модель удаляется из памяти, CUDA cache очищается, затем печатается список созданных topic-файлов в рабочей директории Kaggle.


In [ ]:
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print('Topic notebook outputs:')
for p in sorted(WORKING.glob('*topic*')):
    print(p, p.stat().st_size)
